In [ ]:
from typing import TypedDict,Optional, List, Any
from langgraph.graph import StateGraph, END


#Estabelecendo estados existentes
class AgentState(TypedDict):
    # Entrada e validação inicial
    pergunta: str
    em_escopo: bool

    # Cache
    resposta_cache: Optional[str]
    
    # SQL e Validação de segurança
    geracao_sql: Optional[str]
    sql_valido: bool
    
    # Execução no SQLite e Auto-correção
    resultado_sql: Optional[List[dict]]
    erro_sql: Optional[str]
    tentativas_correcao: int
    
    #Entrega final
    resposta_final: Optional[str]

In [ ]:
# Estabelecendo arestas do grafo
workflow = StateGraph(AgentState)

workflow.set_entry_point("guardrail_entrada")


def decidir_rumo_guardrail(state : AgentState) -> str:
    if state['em_escopo']:
        return "ir_para_cache"
    else:
        return "ir_para_retorno_amigavel"

workflow.add_conditional_edges(
    "guardrail_entrada",
    decidir_rumo_guardrail,
    {
        "ir_para_cache" : "cache_local",
        "ir_para_retorno_amigavel" : "retorno_amigavel"
    }
)

def decidir_rumo_do_cache(state: AgentState) -> str:
    if state["resposta_cache"]:
        return "ir_para_resposta_imediata"
    else:
        return "ir_para_geracao_sql"

workflow.add_conditional_edges(
    "cache_local",
    decidir_rumo_do_cache,
    {
        "ir_para_resposta_imediata" : "resposta_imediata",
        "ir_para_geracao_sql" : "geracao_sql"
    }
)

workflow.add_edge("geracao_sql", "guardrail_sql")

def decidir_rumo_guardrail_sql(state: AgentState) -> str:
    if state["sql_valido"]:
        return "ir_para_execucao"
    else:
        return "ir_para_execucao_bloqueada"

workflow.add_conditional_edges(
    "guardrail_sql",
    decidir_rumo_guardrail_sql,
    {
        "ir_para_execucao": "sqlite_readonly", 
        "ir_para_execucao_bloqueada": "execucao_bloqueada"
    }
)

def decidir_rumo_execucao(state: AgentState) -> str:
    if state["erro_sql"] and state["tentativas_correcao"] < 1:
        return "ir_para_autocorrecao"
    
    elif state["erro_sql"]:
        return "ir_para_erro_definitivo"
    
    else:
        return "ir_para_sintese"

workflow.add_conditional_edges(
    "sqlite_readonly",
    decidir_rumo_execucao,
    {
        "ir_para_autocorrecao": "auto_correcao",
        "ir_para_erro_definitivo": "retorno_amigavel",
        "ir_para_sintese": "sintese_resposta"
    }
)

workflow.add_edge("auto_correcao", "guardrail_sql")

workflow.add_edge("sintese_resposta", END)
workflow.add_edge("retorno_amigavel", END)
workflow.add_edge("resposta_imediata", END)
workflow.add_edge("execucao_bloqueada", END)